# Lesson 1.2 — The Six-Stage Workflow as Data

The spine is not a slogan; it is a data structure. We read the real `LAYER_REGISTRY` and confirm its order, owners, and per-stage I/O.

In [1]:
from modules.module09.integration import LAYER_REGISTRY, STAGES, stage_info
checks = []
for e in LAYER_REGISTRY:
    print(f"{e['stage']:11s} | owner: {e['module']:24s} | reads {e['reads']} -> writes {e['writes']}")
print('STAGES:', STAGES)


Perceive    | owner: Module 3                 | reads ['camera / world'] -> writes ['state.detections']
Understand  | owner: Module 9 (selection) + M4/M5 | reads ['state.detections'] -> writes ['state.targets', 'state.current_target']
Plan        | owner: Module 7                 | reads ['state.current_target', 'state.q'] -> writes ['state.reference']
Execute     | owner: Module 6 + Module 8      | reads ['state.reference', 'state.q'] -> writes ['state.command', 'state.q']
Track       | owner: Module 8 telemetry       | reads ['state.reference', 'state.q'] -> writes ['state.tracking_error', 'state.health']
Recover     | owner: Module 9 orchestrator    | reads ['state.health', 'state.tracking_error'] -> writes ['state.current_target', 'state.stage']
STAGES: ['Perceive', 'Understand', 'Plan', 'Execute', 'Track', 'Recover']


In [2]:
# 1) exactly six stages in canonical order
checks.append(STAGES == ['Perceive','Understand','Plan','Execute','Track','Recover'])
# 2) Perceive owns detections; Understand owns the target decision
checks.append(any('detections' in w for w in stage_info('Perceive')['writes']))
checks.append(any('current_target' in w for w in stage_info('Understand')['writes']))
# 3) Plan writes the reference; Execute writes the command
checks.append(any('reference' in w for w in stage_info('Plan')['writes']))
checks.append(any('command' in w for w in stage_info('Execute')['writes']))


### Which stage can change a *later* pick?
Only **Recover** can revise `current_target` and route the flow backward. We confirm it is the unique stage whose writes touch `current_target` besides Understand.

In [3]:
writers_of_target = [e['stage'] for e in LAYER_REGISTRY
                     if any('current_target' in w for w in e['writes'])]
print('stages that can set current_target:', writers_of_target)
checks.append('Recover' in writers_of_target and 'Understand' in writers_of_target)
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


stages that can set current_target: ['Understand', 'Recover']
6/6 checks passed.
All checks passed.
